# Módulo 4: RAG · Notebook 2 — Memoria en chatbots

Un LLM es **sin estado**: sin memoria, cada turno empieza de cero. La memoria inyecta contexto previo en el prompt para mantener conversaciones coherentes.

**Objetivos:** conocer los tipos de memoria en LangChain Classic, saber cuándo usar cada uno y probarlos con Ollama.

Referencias: [Medium](https://medium.com/@mail2rajivgopinath/understanding-langchain-memory-and-its-use-cases-37b84d0ddce6) · [OneUptime](https://oneuptime.com/blog/post/2026-01-27-langchain-memory/view) · [LangChain Docs](https://docs.langchain.com/oss/python/concepts/memory)

## 1. ¿Cómo elegir el tipo de memoria?

Usa este árbol de decisión para seleccionar la estrategia adecuada según longitud del contexto, personalización y restricciones de tokens:

![Árbol de decisión — tipos de memoria LangChain](../../docs/images/memory.png)

| Pregunta | Si… | Memoria recomendada |
|----------|-----|---------------------|
| ¿Contexto corto? | **No** | `ConversationSummaryBufferMemory` |
| ¿Personalización? | **No** | `ConversationBufferMemory` |
| ¿Tokens limitados? | **Sí** | `ConversationTokenBufferMemory` |
| ¿Solo esta sesión? | **Sí** | `ConversationTokenBufferMemory` |
| ¿Memoria entre sesiones? | **No** | `ConversationEntityMemory` |

> **Corto vs. largo plazo:** el historial de la sesión es memoria *corta*; perfiles de usuario o hechos persistentes son memoria *larga* (LangGraph Store). Ver también `chat_memory_console.py` para un chatbot de consola con OpenAI.

## 2. Configuración

```bash
ollama serve && ollama pull llama3.2:1b && ollama pull nomic-embed-text
```

In [9]:
import warnings
warnings.filterwarnings("ignore")

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_classic.memory import (
    ConversationBufferMemory,
    ConversationBufferWindowMemory,
    ConversationSummaryMemory,
    ConversationSummaryBufferMemory,
    ConversationEntityMemory,
    VectorStoreRetrieverMemory,
)
from langchain_classic.chains import ConversationChain
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

llm = ChatOllama(model="llama3.2:1b", temperature=0)
summary_llm = ChatOllama(model="llama3.2:1b", temperature=0)
print("✅ Entorno listo")

✅ Entorno listo


## 3. Ejemplos por tipo

| Clase | Qué hace |
|-------|----------|
| `ConversationBufferMemory` | Guarda todo el historial |
| `ConversationBufferWindowMemory` | Solo las últimas *k* interacciones |
| `ConversationSummaryMemory` | Resumen progresivo con LLM |
| `ConversationSummaryBufferMemory` | Recientes en bruto + resumen del resto |
| `ConversationEntityMemory` | Rastrea entidades (personas, empresas) |
| `VectorStoreRetrieverMemory` | Recupera turnos por similitud semántica |

### Buffer — historial completo

In [10]:
buffer = ConversationBufferMemory()
buffer.save_context({"input": "Me llamo Alice, trabajo en Acme Corp."}, {"output": "Hola Alice."})
buffer.save_context({"input": "¿En qué empresa trabajo?"}, {"output": "En Acme Corp."})
print(buffer.load_memory_variables({})["history"])

Human: Me llamo Alice, trabajo en Acme Corp.
AI: Hola Alice.
Human: ¿En qué empresa trabajo?
AI: En Acme Corp.


### Ventana — últimas k interacciones

In [11]:
window = ConversationBufferWindowMemory(k=2)
for u, a in [("Me llamo Charlie.", "Hola Charlie."), ("Vivo en Madrid.", "Qué bien."),
             ("Soy ingeniero.", "Interesante."), ("Mi lenguaje favorito es Python.", "Genial.")]:
    window.save_context({"input": u}, {"output": a})
print(window.load_memory_variables({})["history"])

Human: Soy ingeniero.
AI: Interesante.
Human: Mi lenguaje favorito es Python.
AI: Genial.


### Resumen y híbrido

In [12]:
summary_mem = ConversationSummaryMemory(llm=summary_llm)
summary_mem.save_context({"input": "Problema con pedido #12345."}, {"output": "Revisemos el pedido."})
summary_mem.save_context({"input": "Llegó rojo, pedí azul."}, {"output": "Lamento el error."})
print("--- Resumen ---")
print(summary_mem.load_memory_variables({})["history"][:400], "...")

hybrid = ConversationSummaryBufferMemory(llm=summary_llm, max_token_limit=80)
for msg in ["Reunión 10am", "Invitar a María", "Tema: proyecto RAG"]:
    hybrid.save_context({"input": msg}, {"output": f"Anotado: {msg}"})
print("\n--- Híbrido ---")
print(hybrid.load_memory_variables({})["history"][:400], "...")

--- Resumen ---
Here is the progressively summarized text:

The human asks what the AI thinks of artificial intelligence. The AI thinks artificial intelligence is a force for good because it will help humans reach their full potential.

The human asks why do you think artificial intelligence is a force for good? The AI responds that artificial intelligence will help humans reach their full potential.

The human a ...

--- Híbrido ---
Human: Reunión 10am
AI: Anotado: Reunión 10am
Human: Invitar a María
AI: Anotado: Invitar a María
Human: Tema: proyecto RAG
AI: Anotado: Tema: proyecto RAG ...


### Entidades y memoria vectorial

In [13]:
entity_mem = ConversationEntityMemory(llm=llm)
entity_mem.save_context({"input": "Trabajo con Sarah y Mike en TechCorp."}, {"output": "Entendido."})
entity_mem.save_context({"input": "Sarah es CTO, prefiere Python."}, {"output": "Registrado."})
for name, desc in entity_mem.entity_store.store.items():
    print(f"• {name}: {desc}")

emb = OllamaEmbeddings(model="nomic-embed-text:latest")
vs = Chroma.from_documents([Document(page_content="init")], emb, collection_name="mem_demo")
vector_mem = VectorStoreRetrieverMemory(retriever=vs.as_retriever(search_kwargs={"k": 2}))
vector_mem.save_context({"input": "Mi color favorito es azul."}, {"output": "Anotado."})
vector_mem.save_context({"input": "Estudio ML desde hace 6 meses."}, {"output": "¡Bien!"})
print("\n--- Recuperación semántica (color) ---")
print(vector_mem.load_memory_variables({"prompt": "¿Qué color me gusta?"}))


--- Recuperación semántica (color) ---
{'history': 'input: Mi color favorito es azul.\noutput: Anotado.\ninput: Estudio ML desde hace 6 meses.\noutput: ¡Bien!'}


## 4. Chatbot con ConversationChain

In [14]:
chat = ConversationChain(
    llm=llm,
    memory=ConversationBufferWindowMemory(k=4),
    verbose=False,
)
print(chat.predict(input="Hola, soy Ana y estudio IA."))
print(chat.predict(input="¿Recuerdas mi nombre y qué estudio?"))

Hola Ana! Me alegra que estés aquí. Estoy en un entorno de aprendizaje automático en una universidad de tecnología en Madrid. Mi entrenamiento se realizó en una base de datos con más de 100.000 registros de conversaciones humanas y puedo entender el lenguaje español con gran precisión. ¿En qué puedo ayudarte hoy?
Lo siento, Ana. Me parece que no tengo esa información en mi base de datos. Mi entrenamiento se centró principalmente en conversaciones sobre tecnología y ciencia de datos, pero no tengo acceso a bases de datos específicas de personas o registros de conversaciones humanas. ¿Podrías decirme tu nombre real?


## 5. Resumen

- Elige memoria según el **árbol de decisión** de arriba.
- **Buffer/ventana** para prototipos; **resumen** para chats largos; **entidad/vector** para personalización avanzada.
- Combina memoria conversacional con **RAG** (`01_basic_rag.ipynb`) para hechos documentales.
- En producción, considera **LangGraph** (checkpointers + Store).